# 06 — Who evaluates the evaluator?

**RAG Evaluation Lab · Northstar Insurance / PolicyAssist**

## Incident

An automated judge gives a high score while underwriters identify material policy omissions.

Your job is not to make a score look good. Diagnose **where** PolicyAssist fails, explain the severity, choose the smallest change that addresses the failure, and prove the change with a regression test.

## Learning outcomes

By the end, you will be able to reason about verbosity, position, reference, prompt, and domain bias; calibration. This notebook contains the theory and the implementation together; the reusable deterministic functions live in `src/rag_evaluation/`.

## Why this matters

A RAG assistant is a multi-stage system: ingestion and governance → candidate retrieval → reranking → context construction → generation → claims and citations → safety → production operations. A final answer score is a lagging signal. It cannot tell you whether the fault was a stale source, poor ranking, a context budget, unsafe access, a lucky model answer, or an unreliable judge.

```mermaid
flowchart LR
  Q[Policy question] --> R[Retrieve candidates]
  R --> K[Rerank]
  K --> C[Build authorized context]
  C --> G[Generate answer]
  G --> V[Verify claims and citations]
  V --> O[Trace, gate, monitor]
```

For the conceptual framework, read [Evaluating RAG systems beyond the demo](https://oneplusi.io/blog/article/evaluating-rag-systems/).

In [ ]:
from src.rag_evaluation.metrics import EvaluationCase, EvaluationTrace, precision_at_k, recall_at_k, reciprocal_rank, ndcg_at_k, claim_support, release_decision
from src.rag_evaluation.policyassist import CORPUS, retrieve, safe_context, simulate_answer

question = "What is the 2026 temporary accommodation limit?"
retrieved = retrieve(question, include_stale=True, k=3)
[(chunk.id, chunk.year, chunk.text) for chunk in retrieved]

## Investigation — make the intermediate state visible

Run the next cell before looking at the final answer. The source IDs, years, and access controls are evidence. In a production trace, preserve the original query, query rewrite, retrieval candidates, ranks, filters, final context, model response, citations, latency, tokens, cost, and evaluator outputs. Without this trace, a bad answer is a mystery rather than a fixable defect.

In [ ]:
context = safe_context(retrieved)
answer, citations = simulate_answer(question, context)
trace = EvaluationTrace(question, [c.id for c in retrieved], [c.id for c in context], {answer: citations}, latency_ms=420, estimated_cost=0.004)
print("answer:", answer)
print("retrieved:", trace.retrieved_ids)
print("context:", trace.context_ids)
print("citations:", trace.citations)

## Metric and diagnosis

Choose a metric that matches the boundary you are evaluating. Retrieval metrics answer whether relevant evidence appeared; context metrics answer what actually reached generation; generation metrics distinguish correctness from groundedness; claim-level citation checks answer whether each material statement is verifiable. Safety and operations are release constraints, not optional polish.

In [ ]:
relevant = {"home-2026-temp"}
ids = trace.retrieved_ids
print({
    "precision@3": round(precision_at_k(ids, relevant, 3), 2),
    "recall@3": round(recall_at_k(ids, relevant, 3), 2),
    "mrr": round(reciprocal_rank(ids, relevant), 2),
    "ndcg@3": round(ndcg_at_k(ids, {"home-2026-temp": 3, "home-2025-temp": 1}, 3), 2),
})

## Experiment — introduce a controlled failure

1. Run the retrieval with `include_stale=True` and then `False`.
2. Change `k` from 1 to 5. Record recall, noise, and context size.
3. Ask an unanswerable 2027 question. The correct behavior is an abstention, not a plausible forecast.
4. Ask as a support user for `fraud-risk`; the restricted chunk must not be retrieved.

Write down: **failure boundary → severity → hypothesis → metric → smallest improvement → regression case**.

In [ ]:
for include_stale in (True, False):
    hits = retrieve(question, include_stale=include_stale, k=3)
    print({"include_stale": include_stale, "ids": [h.id for h in hits], "recall@3": recall_at_k([h.id for h in hits], relevant, 3)})

metrics = {"recall_at_10": 0.94, "citation_support": 0.93, "abstention_accuracy": 0.72, "permission_leak_rate": 0.0}
print("release decision:", release_decision(metrics))

## Production guidance and reflection

- Keep deterministic checks deterministic: authorization, source version, source ID, latency budget, and required citations do not need a model judge.
- Calibrate LLM judges against expert labels; test verbosity, position, reference, prompt, and domain bias.
- Slice every metric by question type, freshness, role, severity, and document family. A high average can conceal a critical failure.
- Gate high-severity regressions in CI and monitor traces, user corrections, escalations, drift, and cost after release.

### References

- [One+i evaluation guide](https://oneplusi.io/blog/article/evaluating-rag-systems/)
- [RAGAS](https://arxiv.org/abs/2309.15217)
- [RAGBench](https://arxiv.org/abs/2407.11005)
- [RAGChecker](https://arxiv.org/abs/2408.08067)

**Reflection:** Which single metric could look healthy while this system remains unsafe or unusable? What trace field would reveal the real fault?